In [1]:
import sys                      # Módulo para inspeccionar el intérprete en ejecución
print(sys.executable)           # Debe apuntar a .../.venv/bin/python; si no, cambia el kernel

/home/benmp/dev/pln/nlp-text-normalization-spanish/.venv/bin/python


In [2]:
from nltk import ngrams
from collections import defaultdict, Counter
import random
import re
import numpy as np

corpus = [
    "el gato está en el tejado",
    "el perro está en el jardín",
    "el gato come en el jardín",
    "el perro come en el tejado"
]

In [3]:
def preprocesar_para_ngrama(texto):
    texto = texto.lower().strip()
    texto = re.sub(r'[^\w\s]', '', texto)  # solo quitar puntuación
    # NO lematizar agresivamente, NO quitar stopwords
    # porque el contexto secuencial importa
    return texto.split()

In [4]:
INICIO, FIN, DESCONOCIDA = "<s>", "</s>", "<unk>"


def rellenar(tokens, n):
    # n-1 marcas de inicio: el primer token necesita un contexto completo.
    # UNA sola marca de fin: </s> es un evento que se predice una vez por frase.
    # nltk.ngrams(pad_right=True) agrega n-1 marcas, y con N=3 inventa el
    # evento ('tejado', '</s>') -> '</s>', que no ocurre en el texto
    return [INICIO] * (n - 1) + tokens + [FIN]


def a_vocabulario(tokens, vocabulario):
    # Toda palabra fuera del vocabulario se trata como <unk>
    return [t if t in vocabulario else DESCONOCIDA for t in tokens]


def entrenar_modelo(corpus, n=2, suavizado=1, frecuencia_minima=1):
    frases = [preprocesar_para_ngrama(f) for f in corpus]

    # El vocabulario es lo que el modelo puede PREDECIR: las palabras con
    # frecuencia suficiente, más </s> (el fin de frase también se predice) y
    # <unk> (la masa reservada para lo no visto). <s> no entra: solo aparece
    # como contexto, nunca se predice.
    # Con frecuencia_minima=1 ninguna palabra de entrenamiento pasa a <unk> y
    # <unk> recibe solo la masa del suavizado; con 2 o más, las palabras raras
    # se reemplazan por <unk> y su probabilidad se estima con conteos reales
    frecuencias = Counter(t for tokens in frases for t in tokens)
    vocabulario = {t for t, c in frecuencias.items() if c >= frecuencia_minima}
    vocabulario |= {FIN, DESCONOCIDA}

    modelo = defaultdict(Counter)
    for tokens in frases:
        for grama in ngrams(rellenar(a_vocabulario(tokens, vocabulario), n), n):
            contexto = grama[:-1]
            siguiente = grama[-1]
            modelo[contexto][siguiente] += 1

    # El suavizado se aplica aquí, al momento de calcular las probabilidades
    # y se guarda en el modelo, no se pasa como parámetro después
    probabilidades = defaultdict(dict)
    for contexto, conteos in modelo.items():
        total = sum(conteos.values()) + suavizado * len(vocabulario)
        for palabra in vocabulario:
            probabilidades[contexto][palabra] = (conteos[palabra] + suavizado) / total

    return probabilidades, vocabulario


def probabilidad(probabilidades, vocabulario, contexto, palabra):
    palabra = palabra if palabra in vocabulario else DESCONOCIDA
    if contexto in probabilidades:
        return probabilidades[contexto][palabra]
    # Contexto nunca visto: con Laplace todos sus conteos son 0 y la fórmula
    # da (0 + k) / (0 + k·|V|) = 1/|V|. No es un caso especial: es la misma
    # fórmula con conteos en cero
    return 1 / len(vocabulario)

In [5]:
# Ahora predecir y calcular perplejidad solo consumen probabilidades
def predecir_siguiente(probabilidades, contexto):
    if contexto not in probabilidades:
        return []
    # Desempate alfabético: el vocabulario es un set y su orden de iteración
    # cambia entre ejecuciones (hash aleatorio de str), igual que los empates
    return sorted(probabilidades[contexto].items(), key=lambda x: (-x[1], x[0]))[:6]



# --- Comparar distintos valores de N ---
for n in [2, 3]:
    print(f"\n{'='*40}")
    print(f"Modelo con N={n}")
    modelo, vocab = entrenar_modelo(corpus, n=n)

    contexto = ("el", ) if n == 2 else ("el", "gato")[-n+1:]
    resultado = predecir_siguiente(modelo, contexto)

    print(f"Contexto: {contexto}")
    for palabra, prob in resultado:
        print(f"  P({palabra} | contexto) = {prob:.3f}")


Modelo con N=2
Contexto: ('el',)
  P(gato | contexto) = 0.167
  P(jardín | contexto) = 0.167
  P(perro | contexto) = 0.167
  P(tejado | contexto) = 0.167
  P(</s> | contexto) = 0.056
  P(<unk> | contexto) = 0.056

Modelo con N=3
Contexto: ('el', 'gato')
  P(come | contexto) = 0.167
  P(está | contexto) = 0.167
  P(</s> | contexto) = 0.083
  P(<unk> | contexto) = 0.083
  P(el | contexto) = 0.083
  P(en | contexto) = 0.083


In [6]:
# Cada distribución P(· | contexto) debe sumar 1. Con </s> fuera del
# vocabulario, los contextos que preceden al fin de frase ('tejado', 'jardín')
# sumaban 0.8: </s> se contaba en el denominador pero nunca recibía probabilidad
for n in [2, 3]:
    modelo, vocab = entrenar_modelo(corpus, n=n)
    for contexto, distribucion in modelo.items():
        suma = sum(distribucion.values())
        assert abs(suma - 1) < 1e-9, f"N={n}, contexto {contexto}: suma {suma}"
    print(f"N={n}: {len(modelo)} contextos, todas las distribuciones suman 1 · |V| = {len(vocab)}")

N=2: 9 contextos, todas las distribuciones suman 1 · |V| = 10
N=3: 13 contextos, todas las distribuciones suman 1 · |V| = 10


In [7]:
def perplejidad(probabilidades, vocabulario, frases, n):
    # PP = exp(-(1/N) · Σ log P(w_i | contexto))
    # N cuenta cada token predicho: incluye </s> y excluye <s>, que nunca se
    # predice. Se acumula en logaritmos porque el producto de muchas
    # probabilidades menores que 1 cae por debajo del menor float representable
    log_prob, predichos = 0.0, 0
    for frase in frases:
        tokens = rellenar(a_vocabulario(preprocesar_para_ngrama(frase), vocabulario), n)
        for grama in ngrams(tokens, n):
            log_prob += np.log(probabilidad(probabilidades, vocabulario, grama[:-1], grama[-1]))
            predichos += 1
    return float(np.exp(-log_prob / predichos))

In [8]:
# Referencia: un modelo uniforme asigna 1/|V| a todo y su perplejidad es |V|.
# Un modelo que aprendió algo debe quedar por debajo en texto parecido al de
# entrenamiento, y acercarse o pasarse de |V| en texto que contradice el orden
pruebas = {
    "entrenamiento": corpus,
    "recombinada (n-gramas vistos)": ["el gato está en el jardín"],
    "palabra nueva (ratón → <unk>)": ["el ratón come en el tejado"],
    "desordenada": ["tejado el en está gato el"],
}

for n in [2, 3]:
    modelo, vocab = entrenar_modelo(corpus, n=n)
    # Control de la fórmula: sin ningún contexto aprendido, todo vale 1/|V|
    assert abs(perplejidad({}, vocab, corpus, n) - len(vocab)) < 1e-9
    print(f"\nN={n} · |V|={len(vocab)} · modelo uniforme: PP={len(vocab)}")
    for nombre, frases in pruebas.items():
        print(f"  {nombre:<32} PP = {perplejidad(modelo, vocab, frases, n):6.2f}")


N=2 · |V|=10 · modelo uniforme: PP=10
  entrenamiento                    PP =   4.30
  recombinada (n-gramas vistos)    PP =   4.30
  palabra nueva (ratón → <unk>)    PP =   5.41
  desordenada                      PP =  14.08

N=3 · |V|=10 · modelo uniforme: PP=10
  entrenamiento                    PP =   4.41
  recombinada (n-gramas vistos)    PP =   4.41
  palabra nueva (ratón → <unk>)    PP =   6.04
  desordenada                      PP =  10.49
